# GraphRAG Full Inspection Notebook

**Experiment 1: Austrian Photovoltaic Strategy 2024**

This notebook reads the files already created in `output/`. It does **not** call OpenRouter and does **not** rerun GraphRAG.


In [1]:
from pathlib import Path
import json
import re
import pandas as pd

ROOT = Path.cwd()
OUTPUT = ROOT / "output"
INSPECTION = ROOT / "inspection"
INSPECTION.mkdir(exist_ok=True)

print("Project folder:", ROOT)
print("Output folder exists:", OUTPUT.exists())


Project folder: c:\Users\HP-ZBOOK i7\Downloads
Output folder exists: False


## 1. Load all GraphRAG output files


In [2]:
files = {
    "documents": "documents.parquet",
    "text_units": "text_units.parquet",
    "entities": "entities.parquet",
    "relationships": "relationships.parquet",
    "communities": "communities.parquet",
    "community_reports": "community_reports.parquet",
}

tables = {}
for name, filename in files.items():
    path = OUTPUT / filename
    if path.exists():
        tables[name] = pd.read_parquet(path)
        print(f"{name:20s}: {len(tables[name])} rows")
    else:
        tables[name] = None
        print(f"{name:20s}: FILE NOT FOUND")


documents           : FILE NOT FOUND
text_units          : FILE NOT FOUND
entities            : FILE NOT FOUND
relationships       : FILE NOT FOUND
communities         : FILE NOT FOUND
community_reports   : FILE NOT FOUND


## 2. Indexing statistics


In [3]:
stats_path = OUTPUT / "stats.json"
if stats_path.exists():
    with open(stats_path, "r", encoding="utf-8") as f:
        stats = json.load(f)
    print("Documents:", stats.get("num_documents"))
    print("Total runtime (seconds):", round(stats.get("total_runtime", 0), 2))
else:
    print("stats.json not found")


stats.json not found


## 3. Entity type counts


In [4]:
entities = tables["entities"]
entities["type"].value_counts()


TypeError: 'NoneType' object is not subscriptable

## 4. Inspect all entities


In [ ]:
entity_cols = [c for c in [
    "human_readable_id","title","type","description",
    "frequency","degree","text_unit_ids"
] if c in entities.columns]

entity_review = entities[entity_cols].copy()
entity_review["suggested_schema_type"] = ""
entity_review["canonical_name"] = ""
entity_review["keep_in_controlled_kg"] = ""
entity_review["record_linkage_group"] = ""
entity_review["review_notes"] = ""

entity_review


## 5. Top connected entities


In [ ]:
if "degree" in entities.columns:
    display(
        entities.sort_values("degree", ascending=False)[
            [c for c in ["title","type","degree","frequency","description"] if c in entities.columns]
        ].head(30)
    )


## 6. Inspect all relationships


In [ ]:
relationships = tables["relationships"]
relationship_review = relationships.copy()

if (
    {"title","type"}.issubset(entities.columns)
    and {"source","target"}.issubset(relationships.columns)
):
    type_map = entities.drop_duplicates("title").set_index("title")["type"].to_dict()
    relationship_review["source_raw_type"] = relationship_review["source"].map(type_map)
    relationship_review["target_raw_type"] = relationship_review["target"].map(type_map)

relationship_review["evidence_status"] = ""
relationship_review["suggested_relation_type"] = ""
relationship_review["keep_in_controlled_kg"] = ""
relationship_review["review_notes"] = ""

relationship_review


## 7. Highest-weight relationships


In [ ]:
cols = [c for c in ["source","target","description","weight"] if c in relationships.columns]
relationships.sort_values("weight", ascending=False)[cols].head(50)


## 8. Possible duplicate / alias candidates


In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return ""
    text = str(value).upper().strip()
    text = re.sub(r"[^\wÄÖÜẞ]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

duplicate_check = entities.copy()
duplicate_check["normalized_title"] = duplicate_check["title"].apply(normalize_name)
possible_duplicates = duplicate_check[
    duplicate_check["normalized_title"].duplicated(keep=False)
]

possible_duplicates.sort_values("normalized_title")


## 9. Community reports


In [ ]:
community_reports = tables["community_reports"]
community_reports


## 10. Export inspection CSV files for Excel


In [ ]:
exports = {
    "01_entities_all.csv": entities,
    "02_entity_review.csv": entity_review,
    "03_relationships_all.csv": relationships,
    "04_relationship_review.csv": relationship_review,
    "05_communities_all.csv": tables["communities"],
    "06_community_reports_all.csv": community_reports,
    "07_documents_all.csv": tables["documents"],
    "08_text_units_all.csv": tables["text_units"],
    "09_possible_duplicates.csv": possible_duplicates,
}

for filename, df in exports.items():
    if df is not None:
        df.to_csv(INSPECTION / filename, index=False, encoding="utf-8-sig")
        print("Created:", filename)

print("\nInspection folder:", INSPECTION)


## What we review manually next

For **entities**, compare GraphRAG's raw type with your controlled schema:
`COUNTRY`, `ORGANIZATION`, `TECHNOLOGY`, `INFRASTRUCTURE`, `POLICY`,
`SUPPORT_SCHEME`, `CONSTRAINT`, `MARKET_METRIC`, `TARGET`.

For **relationships**, classify each important edge as source-supported,
inferred, questionable, or rejected before loading it into the controlled KG.
